# Full EEG pipeline + FBCCA

Steps:
1. Load BCI2000 recording
2. Bandpass filter [1–55 Hz]
3. Re-reference to average
4. ICA (extended Infomax)
5. ICLabel — classify components
6. Inspect ICA components
7. Remove artifact ICs
8. Before / after comparison
9. Create epochs [0–3 s from stim onset]
10. FBCCA — compute score per epoch
11. Plot FBCCA stream

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path("../..").resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from eeg.io import load_bci2k
from eeg.inspect import print_summary
from eeg.preprocessing import (
    bandpass_filter,
    rereference_average,
    run_ica,
    label_components,
    remove_artifacts,
    make_epochs,
)
from eeg.fbcca import run_fbcca
from eeg.viz import (
    plot_stim_channel,
    plot_channels,
    plot_channels_psd,
    plot_ica_components,
    plot_before_after,
    plot_fbcca_stream,
)

DATA_FILE = PROJECT_ROOT / "data/eeg/raw/MET000bGridFixedS001R02.dat"

## 1. Load

In [ ]:
raw = load_bci2k(DATA_FILE)
print_summary(raw)

In [ ]:
plot_stim_channel(raw, duration=600)

## 2. Bandpass filter

In [ ]:
L_FREQ = 1.0
H_FREQ = 55.0
FILTER_ORDER = 4

bandpass_filter(raw, l_freq=L_FREQ, h_freq=H_FREQ, order=FILTER_ORDER)

## 3. Re-reference to average

In [ ]:
rereference_average(raw)

In [ ]:
# Spot-check: occipital channels after filtering + reref
CHANNELS = ["Oz", "O1", "O2"]

plot_channels(raw, CHANNELS, duration=10.0)

In [ ]:
plot_channels_psd(raw, CHANNELS, fmax=55.0)

## 4. ICA

Takes ~60–90 seconds.

In [ ]:
N_COMPONENTS = 24

ica = run_ica(raw, n_components=N_COMPONENTS)
print(ica)

## 5. ICLabel — classify components

In [ ]:
labels = label_components(raw, ica)

for i, (label, proba) in enumerate(zip(labels["labels"], labels["y_pred_proba"])):
    print(f"IC {i:02d}: {label} ({proba.max():.0%})")

## 6. Inspect ICA components

Edit `COMPONENT_INDICES` to examine any subset. Look for eye blinks (large frontal topography + slow time course) and muscle artifacts (high-frequency spectrum + peripheral topography).

In [ ]:
COMPONENT_INDICES = [0, 1, 2, 3, 4]

plot_ica_components(raw, ica, COMPONENT_INDICES, labels=labels)

## 7. Remove artifact ICs

Components are excluded automatically when ICLabel probability exceeds the thresholds below. Adjust thresholds if you want to be more or less conservative.

In [ ]:
EYE_THRESHOLD    = 0.7   # exclude eye blink ICs above this probability
MUSCLE_THRESHOLD = 0.5   # exclude muscle artifact ICs above this probability

raw_clean, excluded = remove_artifacts(
    raw, ica, labels,
    eye_threshold=EYE_THRESHOLD,
    muscle_threshold=MUSCLE_THRESHOLD,
)

print(f"Excluded ICs: {excluded}")
print(f"Labels of excluded ICs: {[labels['labels'][i] for i in excluded]}")

## 8. Before / after comparison

In [ ]:
COMPARE_CHANNELS = ["Cz", "Fz", "Pz"]

plot_before_after(raw, raw_clean, COMPARE_CHANNELS, duration=10.0)

## 9. Create epochs

Epochs are locked to rising edges of `STI 014` (DigitalInput1), default window 0–3 s.

In [ ]:
TMIN = 0.0   # s relative to stim onset
TMAX = 3.0   # s relative to stim onset

epochs = make_epochs(raw_clean, tmin=TMIN, tmax=TMAX)
print(epochs)
print(f"Shape: {epochs.get_data().shape}  (trials × channels × samples)")

## 10. FBCCA — compute score per epoch

`run_fbcca` applies 5 filter banks ([5–55], [15–55], [25–55], [35–55], [45–55] Hz), computes the maximum canonical correlation against a sin/cos reference at `FBCCA_FREQ` for each bank, then combines with weights `w_j = j^(−1.25) + 0.25` (Chen 2015).

Set `FBCCA_CHANNELS = None` to use all EEG channels, or provide a list of channel names to restrict the CCA to a spatial subset (e.g. occipital channels for visual SSVEP).

In [ ]:
FBCCA_FREQ     = 10.0          # target stimulus frequency in Hz
FBCCA_CHANNELS = None          # None = all EEG; e.g. ["Oz", "O1", "O2"] for occipital only
N_HARMONICS    = 3

scores = run_fbcca(
    epochs,
    freq=FBCCA_FREQ,
    channels=FBCCA_CHANNELS,
    n_harmonics=N_HARMONICS,
)

print(f"n_epochs : {len(scores)}")
print(f"min      : {scores.min():.4f}")
print(f"max      : {scores.max():.4f}")
print(f"mean     : {scores.mean():.4f}")
print(f"std      : {scores.std():.4f}")

## 11. Plot FBCCA stream

Each point is one epoch (stimulus). x-axis = stimulus number (1-indexed), y-axis = weighted FBCCA score.

In [ ]:
plot_fbcca_stream(scores, freq=FBCCA_FREQ)